# Mutation count probabilities
So far, we have only looked at statistics based on branch lengths of the coalescent tree. However, when dealing with short sequences, we may not have enough mutations to compute stable branch-length-based summary statistics. Instead, we may obtain the distribution of mutational counts. It is particularly informative to consider the SFS computed over small non-recombining blocks.

In the following example we obtain the first 8 unfolded mutational configurations and their probabilities under a three-epoch size-change demography, with a mutation rate of 1 per unit of branch length. Configurations, their probabilities and the order in which {meth}`UnfoldedSFSDistribution.get_mutation_configs() <phasegen.distributions.UnfoldedSFSDistribution.get_mutation_configs>` yields them are described in {meth}`UnfoldedSFSDistribution.get_mutation_config() <phasegen.distributions.UnfoldedSFSDistribution.get_mutation_config>`. Configuration ``(2, 1, 0)`` holds two singletons, one doubleton and no tripletons.

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
import phasegen as pg
import pandas as pd
from itertools import islice

# a three-epoch size-change demography (decline then expansion)
coal = pg.Coalescent(n=4, demography=pg.Demography(pop_sizes={'pop_0': {0: 1, 0.5: 0.3, 1.5: 2}}))

pd.DataFrame(islice(coal.sfs.get_mutation_configs(theta=1), 8))
```

```{glue} python-s0-0-0
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
library(phasegen)
pg <- load_phasegen()

# a three-epoch size-change demography (decline then expansion)
coal <- pg$Coalescent(
    n = 4L,
    demography = pg$Demography(
        pop_sizes = list(pop_0 = 1),
        events = c(
            pg$PopSizeChange(pop = "pop_0", time = 0.5, size = 0.3),
            pg$PopSizeChange(pop = "pop_0", time = 1.5, size = 2)
        )
    )
)

do.call(rbind, reticulate::iterate(
    pg$take_n(coal$sfs$get_mutation_configs(theta = 1), 8L)
))
```

```{glue} r-s0-0-0
```

~~~~~

~~~~~~


The iterator does not terminate, so we consume it until the yielded probability mass, {attr}`UnfoldedSFSDistribution.generated_mass <phasegen.distributions.UnfoldedSFSDistribution.generated_mass>`, exceeds 0.8.

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
it = coal.sfs.get_mutation_configs(theta=1)

# continue until generated mass is above 0.8
pd.DataFrame(pg.takewhile_inclusive(lambda _: coal.sfs.generated_mass < 0.8, it))
```

```{glue} python-s1-0-0
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
it <- coal$sfs$get_mutation_configs(theta = 1)

until <- pg$takewhile_inclusive(function(x) {
    return(coal$sfs$generated_mass < 0.8)
}, it)

# continue until generated mass is above 0.8
do.call(rbind, reticulate::iterate(until))
```

```{glue} r-s1-0-0
```

~~~~~

~~~~~~


The probability of a single configuration is returned by {meth}`UnfoldedSFSDistribution.get_mutation_config() <phasegen.distributions.UnfoldedSFSDistribution.get_mutation_config>`.

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
coal.sfs.get_mutation_config([1, 1, 0], theta=1)
```

```{glue} python-s2-0-0
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
coal$sfs$get_mutation_config(c(1, 1, 0), theta = 1)
```

```{glue} r-s2-0-0
```

~~~~~

~~~~~~


## Layouts
The iterators yield {class}`~phasegen.distributions.MutationConfig` objects, which compare equal to the plain tuple of their counts. Each carries the {class}`~phasegen.distributions.MutationLayout` that defines its bins. A bin merges one or more elementary frequency classes, listed in {attr}`MutationLayout.bins <phasegen.distributions.MutationLayout.bins>`, and {meth}`MutationConfig.to_array() <phasegen.distributions.MutationConfig.to_array>` places the counts at the positions of the classes in the spectrum array.

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
config, p = next(coal.sfs.get_mutation_configs(theta=1))

config.layout.bins, config.to_array()
```

```{glue} python-s3-0-0
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
# a configuration converts to an integer vector of its counts, so it is retrieved unconverted
builtins <- reticulate::import_builtins(convert = FALSE)
config <- reticulate::py_get_item(builtins$`next`(coal$sfs$get_mutation_configs(theta = 1)), 0L)

list(config$layout$bins, reticulate::py_to_r(config$to_array()))
```

```{glue} r-s3-0-0
```

~~~~~

~~~~~~


The iterators and {meth}`UnfoldedSFSDistribution.get_mutation_config() <phasegen.distributions.UnfoldedSFSDistribution.get_mutation_config>` accept any layout of the spectrum. The folded layout of {meth}`UnfoldedSFSDistribution.mutation_layout() <phasegen.distributions.UnfoldedSFSDistribution.mutation_layout>` merges the classes ``i`` and ``n - i``. It is the default layout of {attr}`Coalescent.fsfs <phasegen.distributions.Coalescent.fsfs>`, whose configuration ``(2, 1)`` holds two singletons or tripletons and one doubleton.

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
df = pd.DataFrame(islice(coal.fsfs.get_mutation_configs(theta=1), 30))
      
df.plot(kind='bar', x=0, legend=False, xlabel='config');
```

```{glue} python-s4-0-0
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
df <- do.call(rbind, reticulate::iterate(
    pg$take_n(coal$fsfs$get_mutation_configs(theta = 1), 30L)
))

heights <- as.numeric(df[, 2])
labels <- sapply(df[, 1], function(x) paste(unlist(x), collapse = ", "))

par(mar = c(4.5, 4, 1, 1))
barplot(heights, names.arg = labels, las = 2, xlab = "config", cex.names = 0.6, col = "#1f77b4", border = NA)
```

```{glue} r-s4-0-0
```

~~~~~

~~~~~~


With several demes, ``demes=True`` resolves each bin by the deme in which the mutation occurs. The bins are ordered by deme and then by frequency class.

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
# two demes with asymmetric migration
coal2 = pg.Coalescent(
    n={'pop_0': 2, 'pop_1': 2},
    demography=pg.Demography(
        pop_sizes={'pop_0': {0: 0.5}, 'pop_1': {0: 2}},
        migration_rates={('pop_0', 'pop_1'): 1, ('pop_1', 'pop_0'): 0.2},
    ),
)

layout = coal2.sfs.mutation_layout(demes=True)

pd.DataFrame(islice(coal2.sfs.get_mutation_configs(theta=0.5, layout=layout), 6))
```

```{glue} python-s5-0-0
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
# two demes with asymmetric migration
coal2 <- pg$Coalescent(
    n = list(pop_0 = 2L, pop_1 = 2L),
    demography = pg$Demography(
        pop_sizes = list(pop_0 = 0.5, pop_1 = 2),
        events = c(
            pg$MigrationRateChange(source = "pop_0", dest = "pop_1", time = 0, rate = 1),
            pg$MigrationRateChange(source = "pop_1", dest = "pop_0", time = 0, rate = 0.2)
        )
    )
)

layout <- coal2$sfs$mutation_layout(demes = TRUE)

do.call(rbind, reticulate::iterate(
    pg$take_n(coal2$sfs$get_mutation_configs(theta = 0.5, layout = layout), 6L)
))
```

```{glue} r-s5-0-0
```

~~~~~

~~~~~~


The joint spectrum {attr}`Coalescent.jsfs <phasegen.distributions.Coalescent.jsfs>` has one bin per polymorphic descendant vector, the numbers of descendants a mutated branch subtends in each deme. {meth}`JointSFSDistribution.mutation_layout(folded=True) <phasegen.distributions.JointSFSDistribution.mutation_layout>` merges each descendant vector with its complement. Descendant vectors that no genealogy carries together, such as ``(2, 1)`` and ``(1, 2)``, have probability zero.

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
layout = coal2.jsfs.mutation_layout()

layout.bins
```

```{glue} python-s6-0-0
```

```python
pd.DataFrame(islice(coal2.jsfs.get_mutation_configs(theta=0.5), 6))
```

```{glue} python-s6-1-0
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
layout <- coal2$jsfs$mutation_layout()

reticulate::py_get_attr(layout, "bins")
```

```{glue} r-s6-0-0
```

```r
do.call(rbind, reticulate::iterate(
    pg$take_n(coal2$jsfs$get_mutation_configs(theta = 0.5), 6L)
))
```

```{glue} r-s6-1-0
```

~~~~~

~~~~~~


The two-locus spectrum {attr}`Coalescent.sfs2 <phasegen.distributions.Coalescent.sfs2>` counts the mutations of both loci, with bins labelled ``(locus, i)``, and ``theta`` is the mutation rate per locus. {meth}`TwoLocusSFSDistribution.mutation_layout(loci=(0,)) <phasegen.distributions.TwoLocusSFSDistribution.mutation_layout>` counts the mutations of one locus, whose configuration probabilities equal those of a single-locus coalescent.

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
coal3 = pg.Coalescent(n=3, loci=2, recombination_rate=1)

pd.DataFrame(islice(coal3.sfs2.get_mutation_configs(theta=0.5), 6))
```

```{glue} python-s7-0-0
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
coal3 <- pg$Coalescent(n = 3L, loci = 2L, recombination_rate = 1)

do.call(rbind, reticulate::iterate(
    pg$take_n(coal3$sfs2$get_mutation_configs(theta = 0.5), 6L)
))
```

```{glue} r-s7-0-0
```

~~~~~

~~~~~~
